# Learning Rate Schedules: Warmup, Decay & Restarts

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/learning-rate-schedules)

We implement and plot step / exponential / cosine / warmup+cosine / one-cycle / SGDR schedules, then show on a toy optimization that a schedule beats the best constant rate.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## 1 — The schedule family

In [ ]:
def step_decay(t, eta0=0.1, gamma=0.1, step=30):
    return eta0 * gamma**(t // step)

def exp_decay(t, eta0=0.1, lam=0.03):
    return eta0 * np.exp(-lam*t)

def cosine(t, T, eta0=0.1, eta_min=0.0):
    return eta_min + 0.5*(eta0-eta_min)*(1 + np.cos(np.pi*t/T))

def warmup_cosine(t, T, Tw=10, eta0=0.1, eta_min=0.0):
    if t < Tw:
        return eta0 * t / Tw
    return cosine(t-Tw, T-Tw, eta0, eta_min)

def one_cycle(t, T, eta_max=0.3, eta_start=0.03):
    up = int(0.4*T)
    if t < up:
        return eta_start + (eta_max-eta_start)*(t/up)
    return eta_max * (1 + np.cos(np.pi*(t-up)/(T-up)))/2

T = 100
ts = np.arange(T)
plt.figure(figsize=(10, 5))
plt.plot(ts, [step_decay(t) for t in ts], label='step decay')
plt.plot(ts, [exp_decay(t) for t in ts], label='exponential')
plt.plot(ts, [cosine(t, T) for t in ts], label='cosine annealing')
plt.plot(ts, [warmup_cosine(t, T) for t in ts], label='linear warmup + cosine')
plt.plot(ts, [one_cycle(t, T) for t in ts], label='one-cycle')
plt.xlabel('step / epoch'); plt.ylabel('learning rate')
plt.title('Learning rate schedules'); plt.legend(); plt.tight_layout(); plt.show()

## 2 — SGDR warm restarts

Cosine to the floor, then jump back up with progressively longer cycles.

In [ ]:
def sgdr(total, eta0=0.1, T0=10, mult=2):
    lrs = []
    t, Ti = 0, T0
    while len(lrs) < total:
        for i in range(Ti):
            if len(lrs) >= total: break
            lrs.append(cosine(i, Ti, eta0, 0.0))
        Ti *= mult
    return np.array(lrs)

lrs = sgdr(120)
plt.figure(figsize=(10, 3.5))
plt.plot(lrs, color='#6366f1')
plt.xlabel('step'); plt.ylabel('learning rate')
plt.title('SGDR: cosine warm restarts (cycles 10, 20, 40, 80...)')
plt.tight_layout(); plt.show()

## 3 — A schedule beats the best constant rate

Minimize a simple quadratic bowl; compare constant rates against cosine decay.

In [ ]:
# Loss: f(x) = x^2 (grad 2x). Noisy gradient to mimic SGD.
rng = np.random.default_rng(0)
def run(schedule_fn, T=100, x0=5.0, noise=0.5):
    x = x0
    losses = []
    for t in range(T):
        eta = schedule_fn(t)
        g = 2*x + rng.normal(0, noise)   # noisy gradient
        x -= eta * g
        losses.append(x**2)
    return losses

T = 100
for eta_const in [0.05, 0.3]:
    plt.plot(run(lambda t, e=eta_const: e), label=f'constant {eta_const}')
plt.plot(run(lambda t: cosine(t, T, eta0=0.3, eta_min=0.001)), label='cosine 0.3->0', lw=2, color='#34d399')
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss x^2 (log)')
plt.title('Cosine decay: fast early, stable late'); plt.legend(); plt.tight_layout(); plt.show()
print("Large constant rate is fast but noisy at the end; small is stable but slow; cosine gets both.")

## ✏️ Your turn

**Task A — Transformer inverse-sqrt schedule:** Implement $\eta_t = d^{-0.5}\min(t^{-0.5},\, t\,T_w^{-1.5})$ and plot it; verify the peak occurs exactly at $t = T_w$.

**Task B — Reduce-on-plateau:** Implement a scheduler that tracks a validation loss and multiplies the LR by 0.5 whenever the loss fails to improve for `patience` steps. Feed it a synthetic loss curve that plateaus and show the LR drops occur at the plateaus.

In [ ]:
def transformer_lr(t, d_model=512, warmup=4000):
    # TODO(you): implement the inverse-sqrt schedule (t starts at 1)
    return ...

ts = np.arange(1, 20000)
lrs = [transformer_lr(t) for t in ts]
if lrs[0] is not None:
    peak = ts[np.argmax(lrs)]
    print(f"Peak LR at step {peak} (warmup=4000)")
    plt.plot(ts, lrs, color='#6366f1'); plt.xlabel('step'); plt.ylabel('lr')
    plt.title('Transformer inverse-sqrt schedule'); plt.tight_layout(); plt.show()

<details><summary>Solution — Task A</summary>

```python
def transformer_lr(t, d_model=512, warmup=4000):
    return d_model**-0.5 * min(t**-0.5, t * warmup**-1.5)
# The two branches are equal at t = warmup, so the peak is exactly there.
```
</details>